# Pathway Enrichment - Test Notebook

This notebook tests each component of the pathway enrichment pipeline with partial data.

## 1. Test Treatment Index Streaming

Stream through a small portion of the data to verify treatment discovery works.

In [ ]:
from typing import Any


from datasets import load_dataset
import pandas as pd

print("Testing treatment index building with small sample...")

de_stream = load_dataset(
    "tahoebio/Tahoe-100M",
    name="pseudobulk_differential_expression",
    split="train",
    streaming=True
)

# just test first 500,000 rows (about 1/8 of one file)
ROWS_PER_FILE = 4_000_000
MAX_ROWS = 500_000

treatments = []
seen = set[Any]()

for i, row in enumerate[Any | dict](de_stream):
    if i >= MAX_ROWS:
        break
    
    key = (row['Cell_Name_Vevo'], row['drug'], row['concentration'])
    
    if key not in seen:
        seen.add(key)
        file_num = i // ROWS_PER_FILE
        treatments.append({
            'cell_line': row['Cell_Name_Vevo'],
            'drug': row['drug'],
            'concentration': row['concentration'],
            'file_num': file_num,
            'row_start': i,
        })
    
    if i % 100_000 == 0:
        print(f"  Processed {i:,} rows, found {len(treatments)} treatments...")

print(f"\nFound {len(treatments)} unique treatments in first {MAX_ROWS:,} rows")

Testing treatment index building with small sample...


Resolving data files:   0%|          | 0/3388 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1026 [00:00<?, ?it/s]

  Processed 0 rows, found 1 treatments...


  Processed 100,000 rows, found 2 treatments...


  Processed 200,000 rows, found 4 treatments...


  Processed 300,000 rows, found 5 treatments...


  Processed 400,000 rows, found 7 treatments...



Found 8 unique treatments in first 500,000 rows


In [2]:
# show discovered treatments as a dataframe
treatment_df = pd.DataFrame(treatments)
print(f"Unique cell lines: {treatment_df['cell_line'].nunique()}")
print(f"Unique drugs: {treatment_df['drug'].nunique()}")
print()
treatment_df

Unique cell lines: 1
Unique drugs: 8



,cell_line,drug,concentration,file_num,row_start
0,A549,4EGI-1,0.05,0,0
1,A549,9-ING-41,0.05,0,62710
2,A549,APTO-253,0.05,0,125420
3,A549,AT7519,0.05,0,188130
4,A549,AZD1390,0.05,0,250840
5,A549,AZD2858,0.05,0,313550
6,A549,AZD-7648,0.05,0,376260
7,A549,AZD-8055,0.05,0,438970


## 2. Test Single-File DuckDB Download

Query a specific parquet file to get expression data for one treatment.

In [3]:
import duckdb

print("Testing single-file download with DuckDB...")

# setup duckdb
conn = duckdb.connect()
conn.execute("INSTALL httpfs; LOAD httpfs;")
conn.execute("SET http_timeout=30000;")

# test with file 0 (we know it contains A549 treatments)
base_url = "https://huggingface.co/datasets/tahoebio/Tahoe-100M/resolve/main/metadata/pseudobulk_differential_expression"
url = f"{base_url}/train-00000-of-01026.parquet"

# use the first drug we found: 4EGI-1
drug = treatments[0]['drug']
cell_line = treatments[0]['cell_line']
print(f"\nQuerying file 0 for {drug} in {cell_line}...")

query = f"""
    SELECT * FROM read_parquet('{url}')
    WHERE drug = '{drug}'
      AND Cell_Name_Vevo = '{cell_line}'
      AND ABS(concentration - 0.05) < 0.001
"""

de_data = conn.execute(query).df()
print(f"Retrieved {len(de_data)} rows")
print(f"\nColumns: {list(de_data.columns)}")

Testing single-file download with DuckDB...

Querying file 0 for 4EGI-1 in A549...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Retrieved 62710 rows

Columns: ['gene_name', 'baseMean', 'log2FoldChange', 'lfcSE', 'stat', 'pvalue', 'padj', 'plate', 'n_cells_trt', 'n_cells_ctrl', 'Cell_ID_Cellosaur', 'Cell_ID_DepMap', 'drug', 'concentration', 'concentration_unit', 'Cell_Name_Vevo']


In [4]:
# show sample of expression data
print(f"Sample expression data for {drug} in {cell_line}:")
de_data[['gene_name', 'log2FoldChange', 'padj', 'baseMean']].head(10)

Sample expression data for 4EGI-1 in A549:


,gene_name,log2FoldChange,padj,baseMean
0,TSPAN6,-0.279559,0.756971,16.415594
1,TNMD,NaN,NaN,0.000000
2,DPM1,-0.050576,0.897968,130.059631
3,SCYL3,-0.794847,0.348144,14.038391
4,C1orf112,-0.020508,0.983305,27.576792
5,FGR,3.055907,NaN,0.952125
6,CFH,-0.595210,0.537274,11.456981
7,FUCA2,0.106479,0.892402,33.008358
8,GCLC,-0.196621,0.327625,219.124878
9,NFYA,0.034212,0.964077,49.612526


### Gene Name Analysis

The dataset contains ~62,710 genes per treatment. About 1/3 of these don't have official HGNC gene symbols (they're novel transcripts, pseudogenes, or non-coding RNAs), so the dataset uses their Ensembl ID (ENSG*) as a fallback.

For pathway analysis, we filter these out because:
1. Reactome pathways only contain genes with official symbols
2. ENSG* entries won't match any pathway gene sets
3. This reduces noise and improves analysis quality

In [5]:
# analyze gene naming
total_genes = len(de_data)
ensembl_only = de_data['gene_name'].str.startswith('ENSG', na=False).sum()
has_symbol = total_genes - ensembl_only

print(f"Gene name breakdown:")
print(f"  Total genes: {total_genes:,}")
print(f"  With official symbol: {has_symbol:,} ({100*has_symbol/total_genes:.1f}%)")
print(f"  Ensembl ID only: {ensembl_only:,} ({100*ensembl_only/total_genes:.1f}%)")

# show examples of each
print("\nExamples of genes WITH official symbols:")
display(de_data[~de_data['gene_name'].str.startswith('ENSG', na=False)][['gene_name', 'log2FoldChange', 'padj']].head(5))

print("\nExamples of genes with Ensembl ID only (will be filtered out):")
display(de_data[de_data['gene_name'].str.startswith('ENSG', na=False)][['gene_name', 'log2FoldChange', 'padj']].head(5))

Gene name breakdown:
  Total genes: 62,710
  With official symbol: 41,780 (66.6%)
  Ensembl ID only: 20,930 (33.4%)

Examples of genes WITH official symbols:


,gene_name,log2FoldChange,padj
0,TSPAN6,-0.279559,0.756971
1,TNMD,NaN,NaN
2,DPM1,-0.050576,0.897968
3,SCYL3,-0.794847,0.348144
4,C1orf112,-0.020508,0.983305



Examples of genes with Ensembl ID only (will be filtered out):


,gene_name,log2FoldChange,padj
1598,ENSG00000083622,NaN,NaN
1995,ENSG00000093100,NaN,NaN
2188,ENSG00000100101,0.462519,0.429928
2778,ENSG00000103200,NaN,NaN
3322,ENSG00000106540,NaN,NaN


In [6]:
# show top differentially expressed genes (with official symbols only)
de_with_symbols = de_data[~de_data['gene_name'].str.startswith('ENSG', na=False)]

print(f"Top 10 upregulated genes (official symbols only):")
display(de_with_symbols.nlargest(10, 'log2FoldChange')[['gene_name', 'log2FoldChange', 'padj']])

print(f"\nTop 10 downregulated genes (official symbols only):")
display(de_with_symbols.nsmallest(10, 'log2FoldChange')[['gene_name', 'log2FoldChange', 'padj']])

Top 10 upregulated genes (official symbols only):


,gene_name,log2FoldChange,padj
1942,ESR1,5.432285,NaN
27275,LINC01364,5.240170,NaN
41893,NAV2-AS2,5.239978,NaN
32329,BNC2-AS1,5.239724,NaN
6920,MYBPC3,5.239714,NaN
47074,ROCK1P1,5.017581,NaN
23827,H3C9P,5.017282,NaN
6188,ABHD17A,5.017087,NaN
45192,EEF1A1P22,5.016608,NaN
61879,SORD2P-1,5.016608,NaN



Top 10 downregulated genes (official symbols only):


,gene_name,log2FoldChange,padj
47050,CTNS-AS1,-4.605137,NaN
1902,DLL3,-4.531156,NaN
33109,EGOT,-4.452672,NaN
1622,HAL,-4.370515,NaN
3034,EPHX3,-4.370408,NaN
17858,CD247,-4.327457,NaN
33722,LINC01474,-4.237206,NaN
9349,CACNA2D4,-4.141074,NaN
6002,APOL3,-4.090545,NaN
9966,ANKRD9,-4.089988,NaN


## 3. Test GSEApy Prerank

Run pathway enrichment on the expression data using Reactome pathways.

In [7]:
import gseapy as gp

print(f"Preparing ranked gene list for {drug} in {cell_line}...")

# prepare ranked gene list
ranked = de_data[['gene_name', 'log2FoldChange']].dropna()

# filter out genes without official symbols (ENSG* entries)
# these won't match any pathway gene sets anyway
before_filter = len(ranked)
ranked = ranked[~ranked['gene_name'].str.startswith('ENSG', na=False)]
after_filter = len(ranked)
print(f"Filtered out {before_filter - after_filter} genes without official symbols")

# convert to uppercase (required by Enrichr libraries)
ranked['gene_name'] = ranked['gene_name'].str.upper()
ranked = ranked.drop_duplicates(subset='gene_name')
ranked = ranked.set_index('gene_name')['log2FoldChange']

print(f"Genes in ranked list: {len(ranked)}")
print(f"\nTop 5 upregulated: {ranked.nlargest(5).to_dict()}")
print(f"Top 5 downregulated: {ranked.nsmallest(5).to_dict()}")

Preparing ranked gene list for 4EGI-1 in A549...
Filtered out 6811 genes without official symbols
Genes in ranked list: 21419

Top 5 upregulated: {'ESR1': 5.432285308837891, 'LINC01364': 5.240170478820801, 'NAV2-AS2': 5.239978313446045, 'BNC2-AS1': 5.2397236824035645, 'MYBPC3': 5.2397141456604}
Top 5 downregulated: {'CTNS-AS1': -4.605137348175049, 'DLL3': -4.531156063079834, 'EGOT': -4.452672481536865, 'HAL': -4.370514869689941, 'EPHX3': -4.370408058166504}


In [8]:
print("Running GSEApy prerank (100 permutations for quick test)...")
print("This may take 30-60 seconds...\n")

pre_res = gp.prerank(
    rnk=ranked,
    gene_sets='Reactome_2022',
    min_size=15,
    max_size=500,
    permutation_num=100,  # reduced for testing
    outdir=None,
    seed=42,
    verbose=False,
)

print(f"Got {len(pre_res.res2d)} pathway results")

2026-01-20 16:35:16,195 [WARNING] Duplicated values found in preranked stats: 22.06% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


Running GSEApy prerank (100 permutations for quick test)...
This may take 30-60 seconds...



Got 1142 pathway results


In [18]:
pre_res.res2d.sort_values(by='FDR q-val', ascending=True)

,Name,Term,ES,NES,NOM p-val,FDR q-val,FWER p-val,Tag %,Gene %,Lead_genes
3,prerank,Ras Activation Upon Ca2+ Influx Thru NMDA Rece...,-0.659781,-1.769341,0.0,0.446939,0.53,8/19,20.69%,CAMK2A;LRRC7;CAMK2B;HRAS;LLGL1;GRIN2D;GRIN2B;A...
2,prerank,Metabolism Of Cofactors R-HSA-8978934,-0.599817,-1.784334,0.027778,0.475226,0.45,4/19,11.77%,TACR1;NANOS3;COQ7;DHFR
1,prerank,Metabolism Of Nitric Oxide: NOS3 Activation An...,-0.699346,-1.812399,0.0,0.486541,0.33,2/15,2.04%,TACR1;NANOS3
4,prerank,Interferon Alpha/Beta Signaling R-HSA-909733,-0.472708,-1.705346,0.0,0.615531,0.68,38/59,37.06%,SOCS1;HLA-G;HOXC13;IFITM1;IFI35;OAS2;IFITM2;MX...
0,prerank,Fertilization R-HSA-1187000,-0.637325,-1.817296,0.0,0.905192,0.31,6/16,7.83%,OVGP1;ZP3;ADAM21;HVCN1;IZUMO2;IZUMO1
...,...,...,...,...,...,...,...,...,...,...
381,prerank,"Transport Of Vitamins, Nucleosides, And Relate...",-0.292737,-0.924594,0.655172,1.0,1.0,6/41,7.36%,SLCO2A1;SLC28A1;SLC35A2;SLC29A4;SLCO1C1;SLCO1A2
380,prerank,Initiation Of Nuclear Envelope (NE) Reformatio...,-0.332459,-0.925847,0.5625,1.0,1.0,19/19,66.78%,BANF1;LMNA;KPNB1;CDK1;LEMD2;EMD;VRK1;SIRT2;TMP...
379,prerank,SHC-mediated cascade:FGFR2 R-HSA-5654699,-0.32344,-0.927405,0.6,1.0,1.0,7/19,16.68%,FGF5;FGF17;FGF20;FGF8;HRAS;FGF1;FGF18
385,prerank,IGF1R Signaling Cascade R-HSA-2428924,-0.27019,-0.92405,0.62963,1.0,1.0,12/47,16.68%,FGF19;FGF5;FLT3LG;IGF1;TRIB3;FGF17;FGFR3;FGF20...


In [19]:
# process results
results = pre_res.res2d.copy()
results = results.rename(columns={
    'Term': 'pathway',
    'NES': 'nes',
    'NOM p-val': 'pvalue',
    'FDR q-val': 'fdr',
    'Lead_genes': 'leading_edge',
})

# convert numeric columns
for col in ['nes', 'pvalue', 'fdr']:
    results[col] = pd.to_numeric(results[col], errors='coerce')

results = results[['pathway', 'nes', 'pvalue', 'fdr', 'leading_edge']]
results['drug'] = drug
results['concentration'] = 0.05
results['cell_line'] = cell_line

print(f"Processed {len(results)} pathways")
print(f"\nColumn dtypes:")
print(results.dtypes)

Processed 1142 pathways

Column dtypes:
pathway           object
nes              float64
pvalue           float64
fdr              float64
leading_edge      object
drug              object
concentration    float64
cell_line         object
dtype: object


In [20]:
results

,pathway,nes,pvalue,fdr,leading_edge,drug,concentration,cell_line
0,Fertilization R-HSA-1187000,-1.817296,0.000000,0.905192,OVGP1;ZP3;ADAM21;HVCN1;IZUMO2;IZUMO1,4EGI-1,0.05,A549
1,Metabolism Of Nitric Oxide: NOS3 Activation An...,-1.812399,0.000000,0.486541,TACR1;NANOS3,4EGI-1,0.05,A549
2,Metabolism Of Cofactors R-HSA-8978934,-1.784334,0.027778,0.475226,TACR1;NANOS3;COQ7;DHFR,4EGI-1,0.05,A549
3,Ras Activation Upon Ca2+ Influx Thru NMDA Rece...,-1.769341,0.000000,0.446939,CAMK2A;LRRC7;CAMK2B;HRAS;LLGL1;GRIN2D;GRIN2B;A...,4EGI-1,0.05,A549
4,Interferon Alpha/Beta Signaling R-HSA-909733,-1.705346,0.000000,0.615531,SOCS1;HLA-G;HOXC13;IFITM1;IFI35;OAS2;IFITM2;MX...,4EGI-1,0.05,A549
...,...,...,...,...,...,...,...,...
1137,Apoptosis R-HSA-109581,0.313640,1.000000,1.000000,BMX;LY96;PKP1;SATB1;PROC;DFFB;E2F1;FADD;GAS2;B...,4EGI-1,0.05,A549
1138,RHO GTPases Activate Formins R-HSA-5663220,-0.300987,1.000000,1.000000,CENPM,4EGI-1,0.05,A549
1139,COPI-dependent Golgi-to-ER Retrograde Traffic ...,-0.287393,1.000000,1.000000,KIF6;KDELR3;KIF4B;KIF12;KIFC2;KIF25;KIF5A;NAPG...,4EGI-1,0.05,A549
1140,RHOF GTPase Cycle R-HSA-9035034,0.268272,1.000000,1.000000,POTEF,4EGI-1,0.05,A549


In [10]:
print("Top 10 UPREGULATED pathways (highest NES):")
results.nlargest(10, 'nes')[['pathway', 'nes', 'fdr']]

Top 10 UPREGULATED pathways (highest NES):


,pathway,nes,fdr
3,Azathioprine ADME R-HSA-9748787,1.702007,0.797924
10,Antimicrobial Peptides R-HSA-6803157,1.542782,1.000000
13,Platelet Sensitization By LDL R-HSA-432142,1.532953,1.000000
15,G-protein Beta:Gamma Signaling R-HSA-397795,1.519508,1.000000
20,N-glycan Antennae Elongation In medial/trans-G...,1.485372,1.000000
22,FCERI Mediated Ca+2 Mobilization R-HSA-2871809,1.471440,1.000000
24,Synthesis Of PIPs At Golgi Membrane R-HSA-1660514,1.469109,1.000000
27,Striated Muscle Contraction R-HSA-390522,1.453069,1.000000
33,EGR2 And SOX10-mediated Initiation Of Schwann ...,1.414589,1.000000
34,Synthesis Of PIPs At Plasma Membrane R-HSA-166...,1.412108,1.000000


In [11]:
print("Top 10 DOWNREGULATED pathways (lowest NES):")
results.nsmallest(10, 'nes')[['pathway', 'nes', 'fdr']]

Top 10 DOWNREGULATED pathways (lowest NES):


,pathway,nes,fdr
0,Ras Activation Upon Ca2+ Influx Thru NMDA Rece...,-1.863968,0.612795
1,CREB1 Phosphorylation Thru NMDA Receptor-Media...,-1.758722,0.901684
2,Metabolism Of Nitric Oxide: NOS3 Activation An...,-1.734163,0.723681
4,Synthesis Of Glycosylphosphatidylinositol (GPI...,-1.636967,1.000000
5,Fertilization R-HSA-1187000,-1.620695,1.000000
6,Insertion Of Tail-Anchored Proteins Into Endop...,-1.563940,1.000000
7,Cell-extracellular Matrix Interactions R-HSA-4...,-1.558943,1.000000
8,HS-GAG Degradation R-HSA-2024096,-1.549435,1.000000
9,Interferon Alpha/Beta Signaling R-HSA-909733,-1.544036,1.000000
11,FRS-mediated FGFR3 Signaling R-HSA-5654706,-1.542123,1.000000


In [12]:
# filter for significant pathways
sig_results = results[results['fdr'] < 0.25].sort_values('nes')
print(f"Pathways with FDR < 0.25: {len(sig_results)}")

if len(sig_results) > 0:
    print("\nSignificant pathways:")
    display(sig_results[['pathway', 'nes', 'fdr']])
else:
    print("(No pathways reached FDR < 0.25 - this is normal for some drugs)")

Pathways with FDR < 0.25: 0
(No pathways reached FDR < 0.25 - this is normal for some drugs)


## 4. Save Test Results

Save the enrichment results to a parquet file as a demonstration of the caching mechanism.

In [13]:
from pathlib import Path

# create cache directory
cache_dir = Path("../data/processed/reactome") / cell_line
cache_dir.mkdir(parents=True, exist_ok=True)

# save results
safe_drug = drug.replace("/", "_").replace(" ", "_")
cache_path = cache_dir / f"{safe_drug}_0.05.parquet"
results.to_parquet(cache_path, index=False)

print(f"Saved results to: {cache_path}")
print(f"File size: {cache_path.stat().st_size / 1024:.1f} KB")

Saved results to: ../data/processed/reactome/A549/4EGI-1_0.05.parquet
File size: 122.8 KB


In [14]:
# verify we can reload it
reloaded = pd.read_parquet(cache_path)
print(f"Reloaded {len(reloaded)} rows from cache")
print(f"\nCache file contents:")
reloaded.head()

Reloaded 1142 rows from cache

Cache file contents:


,pathway,nes,pvalue,fdr,leading_edge,drug,concentration,cell_line
0,Ras Activation Upon Ca2+ Influx Thru NMDA Rece...,-1.863968,0.000000,0.612795,CAMK2A;LRRC7;CAMK2B;HRAS;LLGL1;GRIN2D;GRIN2B;A...,4EGI-1,0.05,A549
1,CREB1 Phosphorylation Thru NMDA Receptor-Media...,-1.758722,0.000000,0.901684,CAMK2A;LRRC7;CAMK2B;HRAS;LLGL1;GRIN2D;GRIN2B;A...,4EGI-1,0.05,A549
2,Metabolism Of Nitric Oxide: NOS3 Activation An...,-1.734163,0.043478,0.723681,TACR1;NANOS3,4EGI-1,0.05,A549
3,Azathioprine ADME R-HSA-9748787,1.702007,0.017544,0.797924,SLC28A2;SLC28A3;RNASE1,4EGI-1,0.05,A549
4,Synthesis Of Glycosylphosphatidylinositol (GPI...,-1.636967,0.023256,1.000000,DPM2;PIGP;PIGL;PIGO;PIGV;PIGZ;PIGB,4EGI-1,0.05,A549


## Summary

All pipeline components tested successfully:
1. ✅ Treatment index streaming - discovered treatments from data stream
2. ✅ Single-file DuckDB download - retrieved expression data efficiently 
3. ✅ Gene filtering - removed ENSG* genes without official symbols
4. ✅ GSEApy prerank - computed pathway enrichment scores
5. ✅ Cache save/load - verified parquet caching works

The full treatment index (required for `compute_enrichment()`) still needs to be built once (30-60 min).

In [15]:
conn.close()
print("Done!")

Done!
